In [21]:
import pandas as pd
from pathlib import Path

chemin_csv = Path("data/titanic_propre.csv")

if not chemin_csv.exists():
    chemin_csv = Path("/Users/ytmaiga/heticdev/exercice_python/exercice-08/data/titanic_propre.csv")

titanic = pd.read_csv(chemin_csv)

print(titanic.head())

   survived  pclass     sex   age  sibsp  parch     fare     deck  embark_town
0         0       3    male  22.0      1      0   7.2500  Unknown  Southampton
1         1       1  female  38.0      1      0  71.2833        C    Cherbourg
2         1       3  female  26.0      0      0   7.9250  Unknown  Southampton
3         1       1  female  35.0      1      0  53.1000        C  Southampton
4         0       3    male  35.0      0      0   8.0500  Unknown  Southampton


In [22]:
taux_survie_par_classe_sexe = (
    titanic.groupby(["pclass", "sex"])["survived"]
    .mean()
    .unstack()
    * 100
).round(2)

display(taux_survie_par_classe_sexe)

sex,female,male
pclass,,
1,96.81,36.89
2,92.11,15.74
3,50.00,13.54


Oui, les femmes ont davantage survécu que les hommes dans chacune des trois classes. L'écart est particulièrement marqué en deuxième et troisième classe.

In [24]:
titanic["tranche_age"] = pd.cut(
    titanic["age"],
    bins=[0, 18, 60, float("inf")],
    labels=["Enfants", "Adultes", "Seniors"],
    right=False
)

taux_survie_par_age = (
    titanic.groupby("tranche_age", observed=False)["survived"]
    .mean()
    * 100
).round(2)

display(taux_survie_par_age)

tranche_age
Enfants    53.98
Adultes    36.44
Seniors    26.92
Name: survived, dtype: float64

Les enfants présentent un taux de survie supérieur à celui des adultes et des seniors, ce qui suggère qu'ils ont été prioritaires lors de l'évacuation.

In [25]:
prix_moyen_par_port = (
    titanic.groupby("embark_town")["fare"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

display(prix_moyen_par_port)

repartition_classes_par_port = (
    pd.crosstab(
        titanic["embark_town"],
        titanic["pclass"],
        normalize="index"
    ) * 100
).round(2)

display(repartition_classes_par_port)

embark_town
Cherbourg      59.95
Southampton    27.24
Queenstown     13.28
Name: fare, dtype: float64

pclass,1,2,3
embark_town,,,
Cherbourg,50.60,10.12,39.29
Queenstown,2.60,3.90,93.51
Southampton,19.97,25.39,54.64


Cherbourg présente le prix moyen des billets le plus élevé. Cela s'explique en partie par une proportion plus importante de passagers de première classe parmi les personnes embarquées dans ce port.

In [26]:
titanic_avec_statut_voyage = titanic.assign(
    statut_voyage=(
        (titanic["sibsp"] == 0) &
        (titanic["parch"] == 0)
    ).map({
        True: "Seul",
        False: "Accompagne"
    })
)

taux_survie_selon_voyage = (
    titanic_avec_statut_voyage
    .groupby("statut_voyage")["survived"]
    .mean()
    * 100
).round(2)

display(taux_survie_selon_voyage)

statut_voyage
Accompagne    50.56
Seul          30.35
Name: survived, dtype: float64

Voyager seul diminuait les chances de survie : les passagers accompagnés avaient un taux de survie supérieur à celui des passagers voyageant seuls.